In [54]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from dotenv import load_dotenv
import pandas as pd

In [55]:
load_dotenv()

api_key = os.getenv("AZURE_KEY")
api_endpoint = os.getenv("AZURE_ENDPOINT")

In [56]:
client = TextAnalyticsClient(endpoint=api_endpoint, credential=AzureKeyCredential(api_key))

In [57]:
data = pd.read_csv("data/reviews_mixed.csv")
texts = data["Text"].to_list()
real_sentiments = data["Sentiment"].to_list()

In [58]:
sentiments = []
for left in range(0, len(texts), 10):
    batch = texts[left:left+10]
    result = client.analyze_sentiment(batch, show_opinion_mining = True)

    for idx, doc in enumerate(result):
        sentiments.append(doc.sentiment)

print(real_sentiments[:10])
print(sentiments[:10])

['negative', 'negative', 'positive', 'negative', 'negative', 'positive', 'negative', 'positive', 'positive', 'negative']
['negative', 'negative', 'positive', 'negative', 'negative', 'positive', 'neutral', 'positive', 'positive', 'negative']


In [59]:
correct_predictions = 0
total_predictions = len(real_sentiments)

sentiment_counts = {}

for i in range(total_predictions):
    real = real_sentiments[i]
    predicted = sentiments[i]

    if real not in sentiment_counts:
        sentiment_counts[real] = { 'correct': 0, 'total': 0 }

    sentiment_counts[real]['total'] += 1

    if real == predicted:
        correct_predictions += 1
        sentiment_counts[real]['correct'] += 1

overall_accuracy = (correct_predictions / total_predictions) * 100
print(f"Overall accuracy: {overall_accuracy:.2f}%")

for sentiment, counts in sentiment_counts.items():
    sentiment_accuracy = (counts['correct'] / counts['total']) * 100
    print(f"{sentiment} - {sentiment_accuracy:.2f}%")


Overall accuracy: 71.50%
negative - 66.90%
positive - 81.54%


In [60]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_binary_bow = CountVectorizer(binary=True)

binary_bow_matrix = vectorizer_binary_bow.fit_transform(texts[:10])

binary_bow_df = pd.DataFrame(binary_bow_matrix.toarray(), columns=vectorizer_binary_bow.get_feature_names_out())

print("Binary Bag of Words Matrix:")
print(binary_bow_df)

Binary Bag of Words Matrix:
   and  are  bathroom  bed  bedroom  been  bit  comfortable  comfy  cover  \
0    0    1         0    1        0     0    0            0      0      0   
1    0    0         0    0        0     0    0            0      0      0   
2    0    0         0    0        0     0    0            1      0      0   
3    0    0         0    0        0     0    0            0      0      1   
4    0    0         1    0        0     0    0            0      0      0   
5    0    0         0    1        0     0    0            0      1      0   
6    0    0         0    0        0     1    0            0      0      0   
7    0    0         0    1        0     0    0            1      0      0   
8    1    0         0    0        0     0    0            1      0      0   
9    0    0         0    1        1     0    1            0      0      0   

   ...  the  thin  time  uncomfortable  unconfortable  very  was  with  work  \
0  ...    1     0     0              0      

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorizer_tfidf.fit_transform(texts)

print("Shape of the TF-IDF matrix:", tfidf_matrix.shape)
print(f"There are {tfidf_matrix.shape[0]} documents (reviews) and a vocabulary of {tfidf_matrix.shape[1]} words.\n")

print("A few words from the vocabulary:", vectorizer_tfidf.get_feature_names_out()[15:30])
print(tfidf_matrix)

Shape of the TF-IDF matrix: (207, 520)
There are 207 documents (reviews) and a vocabulary of 520 words.

A few words from the vocabulary: ['adults' 'advertised' 'advised' 'agreed' 'ahead' 'air' 'aircon' 'allow'
 'amazing' 'ambience' 'amenities' 'amenity' 'ample' 'anda' 'anymore']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 978 stored elements and shape (207, 520)>
  Coords	Values
  (0, 373)	0.3695512302109335
  (0, 167)	0.49860539909480445
  (0, 408)	0.38275040464395166
  (0, 336)	0.5952335351093269
  (0, 46)	0.33767397096791646
  (1, 372)	0.2756943109378091
  (1, 378)	0.5720290620972897
  (1, 131)	0.5462487524162513
  (1, 513)	0.5462487524162513
  (2, 287)	0.7892236534597841
  (2, 93)	0.6141058742754305
  (3, 287)	0.3910682738788359
  (3, 481)	0.3910682738788359
  (3, 328)	0.413790510531698
  (3, 112)	0.413790510531698
  (3, 377)	0.4458157043937739
  (3, 458)	0.3910682738788359
  (4, 372)	0.5345704471830705
  (4, 43)	0.8451239181318275
  (5, 46)	0.39880260983957144
 

In [70]:
import gensim
import numpy as np
import re

tokenized_texts = []
for text in texts:
    cleaned_text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = cleaned_text.lower().split()
    tokenized_texts.append(tokens)

print("Exemplu de text pregătit (primele 2 review-uri):")
print(tokenized_texts[:2])

w2v_model = gensim.models.Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=1)

document_vectors = []
for doc_tokens in tokenized_texts:
    word_vectors = [w2v_model.wv[word] for word in doc_tokens if word in w2v_model.wv]
    
    if word_vectors:
        doc_vector = np.mean(word_vectors, axis=0)
        document_vectors.append(doc_vector)
    else:
        document_vectors.append(np.zeros(w2v_model.vector_size))

word2vec_feature_matrix = np.array(document_vectors)

print("Forma matricei finale de feature-uri (Word2Vec):", word2vec_feature_matrix.shape)
print(f"Avem {word2vec_feature_matrix.shape[0]} review-uri, fiecare reprezentat de un vector cu {word2vec_feature_matrix.shape[1]} valori.")

print("\nVectorul (embedding-ul) pentru primul review (primele 10 valori):")
print(word2vec_feature_matrix[0][:10])


Exemplu de text pregătit (primele 2 review-uri):
[['the', 'rooms', 'are', 'extremely', 'small', 'practically', 'only', 'a', 'bed'], ['room', 'safe', 'did', 'not', 'work']]
Forma matricei finale de feature-uri (Word2Vec): (207, 100)
Avem 207 review-uri, fiecare reprezentat de un vector cu 100 valori.

Vectorul (embedding-ul) pentru primul review (primele 10 valori):
[-0.00052722  0.0037075   0.0009225   0.0036836   0.00053528 -0.00263059
  0.00085884  0.00476442 -0.00112262  0.00034231]


In [72]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

X = word2vec_feature_matrix 

y_text = real_sentiments

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} -> {i}")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensiune set antrenare: {X_train.shape[0]} review-uri")
print(f"Dimensiune set testare: {X_test.shape[0]} review-uri")

print("Se antrenează rețeaua neuronală (MLPClassifier)...")

# - hidden_layer_sizes=(100, 50): Două straturi ascunse. Primul cu 100 de neuroni, al doilea cu 50.
# - max_iter = 300: Numărul maxim de epoci (treceri prin datele de antrenare).
# - activation = 'relu': Funcția de activare standard și eficientă.
# - random_state = 42: Pentru reproductibilitatea rezultatelor.
ann_classifier = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, activation='relu', random_state=42)

ann_classifier.fit(X_train, y_train)

print("Evaluarea modelului pe setul de testare:")
y_pred = ann_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Acuratețea modelului: {accuracy * 100:.2f}%")


negative -> 0
positive -> 1
Dimensiune set antrenare: 165 review-uri
Dimensiune set testare: 42 review-uri
Se antrenează rețeaua neuronală (MLPClassifier)...
Evaluarea modelului pe setul de testare:
Acuratețea modelului: 69.05%


In [ ]:
import numpy as np

def to_one_hot(labels, num_classes):
    one_hot_labels = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        one_hot_labels[i, label] = 1
    return one_hot_labels

num_classes = len(np.unique(y))
y_train_one_hot = to_one_hot(y_train, num_classes)
y_test_one_hot = to_one_hot(y_test, num_classes)

class CustomANN:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate

        self.W1 = np.random.randn(self.input_size, self.hidden_size) * 0.01
        self.b1 = np.zeros((1, self.hidden_size))
        self.W2 = np.random.randn(self.hidden_size, self.output_size) * 0.01
        self.b2 = np.zeros((1, self.output_size))

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.softmax(self.z2)
        return self.a2

    def backward(self, X, y_one_hot):
        num_samples = X.shape[0]
        
        delta2 = self.a2 - y_one_hot
        
        dW2 = np.dot(self.a1.T, delta2) / num_samples
        db2 = np.sum(delta2, axis=0, keepdims=True) / num_samples
        
        delta1 = np.dot(delta2, self.W2.T) * (self.a1 * (1 - self.a1))
        
        dW1 = np.dot(X.T, delta1) / num_samples
        db1 = np.sum(delta1, axis=0, keepdims=True) / num_samples
        
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2

    def train(self, X, y_one_hot, epochs):
        for epoch in range(epochs):
            predictions = self.forward(X)
            
            self.backward(X, y_one_hot)
            
            if (epoch % 100) == 0:
                loss = -np.mean(y_one_hot * np.log(predictions + 1e-8))
                print(f"Epoca {epoch}, Cost (Loss): {loss:.4f}")
    
    def predict(self, X):
        probabilities = self.forward(X)
        return np.argmax(probabilities, axis=1)

input_dim = X_train.shape[1]   
hidden_dim = 50                
output_dim = num_classes       
epochs = 1000
learning_rate = 0.01

custom_model = CustomANN(input_size=input_dim, hidden_size=hidden_dim, output_size=output_dim, learning_rate=learning_rate)
print("Începe antrenarea modelului propriu...")
custom_model.train(X_train, y_train_one_hot, epochs=epochs)
print("Antrenare finalizată.\n")

y_pred_custom = custom_model.predict(X_test)
accuracy_custom = np.mean(y_pred_custom == y_test) * 100
print(f"Acuratețea modelului propriu pe setul de testare: {accuracy_custom:.2f}%")

Începe antrenarea modelului propriu...
Epoca 0, Cost (Loss): 0.3422
Epoca 100, Cost (Loss): 0.3116
Epoca 200, Cost (Loss): 0.3116
Epoca 300, Cost (Loss): 0.3116
Epoca 400, Cost (Loss): 0.3116
Epoca 500, Cost (Loss): 0.3116
Epoca 600, Cost (Loss): 0.3116
Epoca 700, Cost (Loss): 0.3116
Epoca 800, Cost (Loss): 0.3116
Epoca 900, Cost (Loss): 0.3116
Antrenare finalizată.

Acuratețea modelului propriu pe setul de testare: 69.05%
